---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md)
*Search [existing issues](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues) first.*

### 💾 Save your own copy
> **File → Save a copy in Drive** — saves a personal editable copy to your Google Drive.
> Do this before making edits, otherwise changes are lost when the session ends.

# 📉 Logistic Regression
**ISLP Chapter 4 · Pattern Recognition for the Rest of Us**

> When the response is binary (default/no default, fraud/not fraud), linear regression predicts probabilities outside [0,1]. Logistic regression solves this with the logistic function — it models log-odds as a linear function of the predictors.

### Dataset: Default (ISLP)
10,000 credit card customers — predict whether a customer will default on their payment.
Classic binary classification benchmark from ISLP Chapter 4.

### What you'll learn
- Why linear regression fails for binary outcomes
- The logistic function and log-odds
- Interpreting coefficients as odds ratios
- Classification metrics: accuracy, precision, recall, F1, AUC
- ROC curve and choosing a decision threshold

### Time: ~50 minutes

In [ ]:
# ⚠️  Run this cell first — sets up data and imports
# Tip: Runtime → Run all  (runs everything top to bottom)

import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay,
                              roc_auc_score, roc_curve, RocCurveDisplay)
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#f8f9fa',
    'axes.grid': True, 'grid.alpha': 0.4,
    'axes.spines.top': False, 'axes.spines.right': False,
    'font.size': 11
})

# ── Load Default dataset (ISLP Chapter 4) ────────────────────────────────────
try:
    default_df = pd.read_csv('https://www.statlearning.com/s/Default.csv')
    print(f'✓ Default.csv (ISLP): {default_df.shape}')
except Exception:
    default_df = pd.read_csv('https://raw.githubusercontent.com/ladataanalytics/pattern-recognition-notebooks/main/data/Default.csv')
    print(f'✓ Default.csv (fallback): {default_df.shape}')

# Binary encode target and student flag
default_df['default_num'] = (default_df['default'] == 'Yes').astype(int)
default_df['student_num'] = (default_df['student'] == 'Yes').astype(int)

print(f'  Default rate: {default_df["default_num"].mean():.2%}')
print(f'  Students:     {default_df["student_num"].mean():.2%}')
print(f'  Balance mean: ${default_df["balance"].mean():.2f}')
print(f'  Income mean:  ${default_df["income"].mean():.2f}')
print(f'  Python {sys.version.split()[0]} | pandas {pd.__version__}')
print('✓ Setup complete')


## 📐 Part 1 — Why Not Linear Regression?

For binary outcomes, linear regression:
- Predicts values outside [0,1] — impossible probabilities
- Assumes a linear relationship between X and P(Y=1) — wrong shape
- Treats a jump from 0.01→0.02 the same as 0.48→0.49 — wrong

**Logistic regression** models the **log-odds** (logit) as a linear function:

```
log(p / (1−p)) = β₀ + β₁X₁ + β₂X₂ + ...
```

Back-transforming gives probabilities that always stay in (0,1):

```
p = 1 / (1 + e^(−(β₀ + β₁X₁ + ...)))
```

In [ ]:
# ── Show why linear regression fails for binary outcomes ─────────────────────
np.random.seed(42)
n = 200
balance = np.sort(np.random.uniform(0, 2500, n))
prob_true = 1 / (1 + np.exp(-(balance - 1200) / 250))
y_bin = (np.random.uniform(0, 1, n) < prob_true).astype(float)

from sklearn.linear_model import LinearRegression
lm = LinearRegression().fit(balance.reshape(-1,1), y_bin)
lr = LogisticRegression().fit(balance.reshape(-1,1), y_bin)

x_range = np.linspace(0, 2500, 300).reshape(-1,1)
p_linear  = lm.predict(x_range)
p_logistic = lr.predict_proba(x_range)[:,1]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, p, title, color in zip(
        axes,
        [p_linear, p_logistic],
        ['Linear Regression — probabilities go outside [0,1]',
         'Logistic Regression — always in (0,1)'],
        ['#e85d20','#1a7a45']):
    ax.scatter(balance, y_bin, alpha=0.3, s=15, color='#1e5fa8',
               label='Observed (0 or 1)')
    ax.plot(x_range, p, color=color, lw=2.5, label='Predicted P(default)')
    ax.axhline(0, color='grey', lw=0.8, ls='--')
    ax.axhline(1, color='grey', lw=0.8, ls='--')
    ax.set_xlabel('Credit Card Balance ($)')
    ax.set_ylabel('P(Default)')
    ax.set_title(title)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()
print("📌 Linear regression predicts negative probabilities for low balances.")
print("   Logistic regression is bounded by the S-shaped logistic function.")


## 🔬 Part 2 — Fitting Logistic Regression (statsmodels)

`statsmodels.Logit` gives the full statistical output: coefficients, standard errors, p-values, and confidence intervals — essential for inference.

**Interpreting coefficients:**
- β is the change in **log-odds** per unit increase in X
- exp(β) is the **odds ratio** — how much the odds multiply per unit increase
- exp(β) > 1 → increased odds of default; exp(β) < 1 → decreased odds

In [ ]:
# ── Fit logistic regression with statsmodels for full statistical output ──────
X_sm = sm.add_constant(default_df[['balance', 'income', 'student_num']])
y_sm = default_df['default_num']

logit_model = sm.Logit(y_sm, X_sm).fit(disp=0)
print(logit_model.summary())

print("\n=== ODDS RATIO INTERPRETATION ===")
coefs = logit_model.params
ci    = logit_model.conf_int()

print(f"\n{'Feature':15s}  {'β':>9}  {'OR=exp(β)':>10}  {'OR 95% CI':>20}  {'p-value':>9}")
print("-" * 72)
for feat in ['balance','income','student_num']:
    b    = coefs[feat]
    OR   = np.exp(b)
    lo   = np.exp(ci.loc[feat, 0])
    hi   = np.exp(ci.loc[feat, 1])
    pval = logit_model.pvalues[feat]
    sig  = '***' if pval<0.001 else '**' if pval<0.01 else '*' if pval<0.05 else 'ns'
    print(f"{feat:15s}  {b:>9.5f}  {OR:>10.4f}  ({lo:.4f}, {hi:.4f})  {pval:>9.4f} {sig}")

print(f"\nPseudo R² (McFadden): {1 - logit_model.llf/logit_model.llnull:.4f}")
print("\n📌 balance is the strongest predictor — each $1 increase multiplies")
print(f"   default odds by {np.exp(coefs['balance']):.5f}")
print(f"   → $100 increase multiplies odds by {np.exp(coefs['balance']*100):.3f}x")
print(f"\n   student_num: being a student multiplies odds by {np.exp(coefs['student_num']):.3f}x")
print("   (controlling for balance — students have lower balances on average)")


## 📊 Part 3 — Prediction, Threshold, and Classification Metrics

The model outputs **probabilities**. We apply a threshold (default 0.5) to convert to class labels.

Changing the threshold trades off:
- **Lower threshold** → more defaults caught (higher recall) but more false alarms (lower precision)
- **Higher threshold** → fewer false alarms but more missed defaults

For credit risk, missing a default (false negative) is usually more costly than a false alarm.

In [ ]:
# ── Train/test split and evaluation ──────────────────────────────────────────
features = ['balance', 'income', 'student_num']
X = default_df[features].values
y = default_df['default_num'].values

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3,
                                            random_state=42, stratify=y)

# Fit sklearn LogisticRegression (for sklearn metrics pipeline)
lr = LogisticRegression(max_iter=1000, random_state=42).fit(X_tr, y_tr)
y_pred      = lr.predict(X_te)
y_prob      = lr.predict_proba(X_te)[:, 1]

print("=== Classification Report (threshold = 0.5) ===")
print(classification_report(y_te, y_pred,
                             target_names=['No Default', 'Default']))

auc = roc_auc_score(y_te, y_prob)
print(f"AUC-ROC: {auc:.4f}")
print()

# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

cm = confusion_matrix(y_te, y_pred)
ConfusionMatrixDisplay(cm, display_labels=['No Default', 'Default'])    .plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Confusion Matrix (threshold = 0.5)\n'
                   f'Accuracy = {accuracy_score(y_te,y_pred):.3f}  '
                   f'AUC = {auc:.3f}')

# ROC curve
fpr, tpr, thresholds = roc_curve(y_te, y_prob)
axes[1].plot(fpr, tpr, color='#1e5fa8', lw=2.5,
             label=f'Logistic Regression (AUC = {auc:.3f})')
axes[1].plot([0,1], [0,1], 'k--', lw=1, label='Random classifier (AUC = 0.5)')
axes[1].fill_between(fpr, tpr, alpha=0.1, color='#1e5fa8')
axes[1].set_xlabel('False Positive Rate (1 − Specificity)')
axes[1].set_ylabel('True Positive Rate (Recall / Sensitivity)')
axes[1].set_title('ROC Curve — Default Dataset')
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()

print("📌 AUC = 1.0 → perfect classifier | AUC = 0.5 → random guessing")
print(f"   AUC = {auc:.3f} means the model ranks a random defaulter above")
print(f"   a random non-defaulter {auc:.1%} of the time.")


## ✅ Part 4 — Decision Threshold and Precision-Recall Tradeoff

AUC summarises performance across all thresholds. For deployment, we must pick one threshold. The right choice depends on the **cost of each error type**:

- **False Negative** (missed default) → bank loses loan amount
- **False Positive** (false alarm) → customer inconvenienced, bank loses business

Plotting precision and recall across thresholds helps find the operating point.

In [ ]:
# ── Threshold analysis ────────────────────────────────────────────────────────
from sklearn.metrics import precision_recall_curve, f1_score

precision, recall, thresh_pr = precision_recall_curve(y_te, y_prob)
f1_scores   = 2 * precision * recall / (precision + recall + 1e-9)
best_f1_idx = np.argmax(f1_scores[:-1])
best_thresh = thresh_pr[best_f1_idx]

# Accuracy, precision, recall, F1 across thresholds
thresholds = np.linspace(0.01, 0.99, 200)
metrics    = {'accuracy':[], 'precision':[], 'recall':[], 'f1':[]}
for t in thresholds:
    y_t = (y_prob >= t).astype(int)
    metrics['accuracy'].append(accuracy_score(y_te, y_t))
    r = classification_report(y_te, y_t, output_dict=True, zero_division=0)
    metrics['precision'].append(r['1']['precision'])
    metrics['recall'].append(r['1']['recall'])
    metrics['f1'].append(r['1']['f1-score'])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for metric, color in zip(['precision','recall','f1','accuracy'],
                          ['#1e5fa8','#e85d20','#1a7a45','#888']):
    axes[0].plot(thresholds, metrics[metric], color=color, lw=2,
                 label=metric.capitalize())
axes[0].axvline(0.5,         color='grey',    lw=1, ls='--', label='Default (0.5)')
axes[0].axvline(best_thresh, color='#c0392b', lw=1.5, ls='--',
                label=f'Best F1 ({best_thresh:.2f})')
axes[0].set_xlabel('Classification Threshold')
axes[0].set_ylabel('Metric value')
axes[0].set_title('Metrics vs Threshold')
axes[0].legend(fontsize=9)

# Precision-Recall curve
axes[1].plot(recall[:-1], precision[:-1], color='#1e5fa8', lw=2.5)
axes[1].scatter(recall[best_f1_idx], precision[best_f1_idx],
                color='#e85d20', s=100, zorder=5,
                label=f'Best F1 = {f1_scores[best_f1_idx]:.3f} @ t={best_thresh:.2f}')
baseline = y_te.mean()
axes[1].axhline(baseline, color='grey', lw=1, ls='--',
                label=f'Random (precision = {baseline:.3f})')
axes[1].set_xlabel('Recall (sensitivity)')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

# Show what changes when we lower threshold to 0.2 (catch more defaults)
y_low = (y_prob >= 0.2).astype(int)
print(f"Threshold 0.50 — defaults caught: {(y_pred[y_te==1]).sum()}/{(y_te==1).sum()}"
      f"  false alarms: {((y_pred==1)&(y_te==0)).sum()}")
print(f"Threshold 0.20 — defaults caught: {(y_low[y_te==1]).sum()}/{(y_te==1).sum()}"
      f"  false alarms: {((y_low==1)&(y_te==0)).sum()}")
print()
print("📌 Lowering the threshold catches more real defaults but triggers")
print("   more false alarms. The right threshold depends on business costs.")
print(f"   Best F1 threshold: {best_thresh:.2f} — balanced precision/recall")


## 💼 Exercise

**Task 1 — Multiclass logistic regression:** The Default dataset is binary. Using a public dataset of your choice (e.g. Iris from sklearn), fit a multinomial logistic regression (`multi_class='multinomial'`). Report per-class precision and recall.

**Task 2 — Calibration:** Plot a calibration curve for the logistic model (`sklearn.calibration.CalibrationDisplay`). Is the model well-calibrated — do predicted probabilities match observed default rates?

**Task 3 — Feature engineering:** Create a new feature `balance_to_income = balance / income`. Does adding it improve AUC?

In [ ]:
# ── Exercise workspace ───────────────────────────────────────────────────────
# Variables: default_df, X_tr, X_te, y_tr, y_te, lr, y_prob

# Task 2 — Calibration curve
from sklearn.calibration import CalibrationDisplay
fig, ax = plt.subplots(figsize=(6, 5))
CalibrationDisplay.from_predictions(y_te, y_prob, n_bins=10,
                                     name='Logistic Regression', ax=ax)
ax.set_title('Calibration Curve — Default Dataset')
plt.tight_layout()
plt.show()
print("Well-calibrated: predicted probabilities match observed default rates")

# Task 3 — Feature engineering
default_df['bal_to_inc'] = default_df['balance'] / (default_df['income'] + 1)
features_new = ['balance', 'income', 'student_num', 'bal_to_inc']
X_new = default_df[features_new].values
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_new, y, test_size=0.3,
                                                random_state=42, stratify=y)
lr2  = LogisticRegression(max_iter=1000, random_state=42).fit(X_tr2, y_tr2)
auc2 = roc_auc_score(y_te2, lr2.predict_proba(X_te2)[:,1])
print(f"\nAUC without balance/income feature: {auc:.4f}")
print(f"AUC with    balance/income feature: {auc2:.4f}")


In [ ]:
# @title 📝 Quiz — Logistic Regression
# @markdown Answer each question, then run this cell.

# @markdown **Q1:** Why can't we use linear regression for binary classification?
# @markdown **Q2:** What does an odds ratio of 1.5 mean for a predictor?
# @markdown **Q3:** A model has AUC = 0.5. What does this mean?
# @markdown **Q4:** You are building a cancer screening model. Would you prefer high precision or high recall? Why?
# @markdown **Q5:** Why does lowering the classification threshold increase recall but decrease precision?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
missing = [k for k,v in answers.items() if not str(v).strip()]
print(f"  {5-len(missing)}/5 answered — run the submission cell for AI feedback")


In [ ]:
_NB_NAME_ = "Logistic Regression"
# @title 📋 Quiz Submission
# @markdown **Click ▶ Run** → copy the output → paste into Gemini in this Colab session.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"your GitHub username e.g. jsmith42"}
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

if "answers" not in globals():
    print("Run the quiz cell above first, then re-run this cell.")
else:
    qa = "\n".join(f"Q{i}: {str(v).strip() or '(no answer)'}"
                    for i, (_, v) in enumerate(answers.items(), 1))
    print(f'Please grade my quiz answers for the "{_NB_TITLE}" notebook')
    print(f'from "Data Pattern Recognition for the Rest of Us" (based on ISLP).')
    if GITHUB_USERNAME.strip(): print(f"Student: @{GITHUB_USERNAME.strip()}")
    print(); print(qa); print()
    print("For each question:")
    print("  1. Say CORRECT, PARTIAL, or INCORRECT")
    print("  2. Explain in 2-3 sentences why")
    print("  3. Give the ideal complete answer")
    print("  4. If wrong/partial, say which Part to revisit")
    print()
    print("End with:")
    print("  - Overall grade: Excellent / Good / Needs Review / Incomplete")
    print("  - A short study plan")


---
### 🐛 Found a bug or have a suggestion?
[Open a GitHub Issue](https://github.com/ladataanalytics/pattern-recognition-notebooks/issues/new?template=bug_report.md)

---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*